# 🎬 VIDEO QUIZ STUDIO — GOOGLE COLAB GPU RUNNER
### Ứng dụng tự động hóa tạo video Quiz (Shorts / Reels / TikTok) chạy trực tiếp trên Google Colab GPU

Sổ tay này giúp bạn chạy toàn bộ giao diện Web Studio, Remotion Native Skia Engine, tổng hợp giọng đọc đa kênh (VieNeu TTS v3 Turbo, Nhân bản giọng Zero-Shot & Microsoft Edge-TTS), render tăng tốc phần cứng NVENC (25 - 35+ FPS) và xuất video MP4 trực tiếp trên nền tảng đám mây của Google Colab.

---
### 📌 Hướng dẫn nhanh:
1. **Bước 1**: Nhập link GitHub vào ô `REPO_URL` (nếu repo Private thì nhập `GITHUB_TOKEN`). Bấm nút Play ▶️ để tải code (siêu nhanh ~3s). *(Mặc định tắt kết nối Google Drive để khởi động nhanh chóng)*.
2. **Bước 1.1 (Tùy chọn)**: Nếu bạn cần **Nhân bản (Clone) giọng mới** hoặc **dùng lại các giọng clone đã lưu**, bấm nút Play ▶️ ở ô **Bước 1.1** để kết nối Google Drive. *Nếu chỉ dùng giọng có sẵn (Edge-TTS / VieNeu AI), bạn KHÔNG CẦN chạy bước này*.
3. **Bước 2**: Bấm nút Play ▶️ để cài đặt môi trường (Đã rút gọn log tối đa, chỉ thông báo khi hoàn tất từng mục).
4. **Bước 3**: Bấm nút Play ▶️ để nhận link Web App HTTPS (Cloudflare Tunnel) và mở Studio sử dụng ngay!


In [ ]:
# @title 📥 BƯỚC 1: Clone / Cập nhật mã nguồn từ GitHub
# @markdown Nhập link GitHub và Token (nếu là repo Private):
REPO_URL = "https://github.com/baomars/Video-quiz-new.git" # @param {type:"string"}
GITHUB_TOKEN = "" # @param {type:"string"}
# @markdown 📂 Tùy chọn Google Drive (Chỉ tích nếu muốn tự động kết nối Drive ngay lập tức):
CONNECT_GOOGLE_DRIVE = False # @param {type:"boolean"}

import os
import subprocess

# 1. Chuẩn hóa link Repository
clean_url = REPO_URL.strip().replace("https://", "").replace("http://", "")
if clean_url.endswith("/"): 
    clean_url = clean_url[:-1]

repo_name = clean_url.split("/")[-1].replace(".git", "")
target_dir = f"/content/{repo_name}"

# 2. Xử lý Token nếu có
token = GITHUB_TOKEN.strip()
if token:
    auth_url = f"https://{token}@{clean_url}"
else:
    auth_url = f"https://{clean_url}"

# 3. Clone hoặc Pull cập nhật
if os.path.exists(target_dir):
    print(f"📂 Thư mục {repo_name} đã tồn tại. Đang đồng bộ cập nhật mới nhất (git pull)...")
    subprocess.run(f"cd '{target_dir}' && git fetch --all && (git reset --hard origin/main || git reset --hard origin/master || git pull origin main || git pull origin master)", shell=True)
    print("\n✅ Đã cập nhật mã nguồn mới nhất!")
else:
    print(f"📥 Đang tải repository {repo_name} về Google Colab...")
    res = subprocess.run(f"git clone {auth_url} '{target_dir}'", shell=True, capture_output=True, text=True)
    if res.returncode == 0:
        print(f"\n✅ Tải thành công vào {target_dir}!")
    else:
        safe_err = res.stderr.replace(token, "********") if token else res.stderr
        print(f"\n❌ Lỗi khi tải repo: {safe_err}")
        print("👉 Vui lòng kiểm tra lại: REPO_URL đã chính xác chưa? Nếu repo là Private, bạn đã điền GITHUB_TOKEN chưa?")

# 4. Bảo mật: Xóa token khỏi remote URL sau khi clone xong
subprocess.run(f"cd '{target_dir}' && git remote set-url origin https://{clean_url}", shell=True)

# 5. Thiết lập biến môi trường và tạo sẵn thư mục làm việc, output & cache
os.environ["PROJECT_DIR"] = target_dir
os.environ["PROJECT_NAME"] = repo_name
for d in ["output", "data/renders", "data/channels", "data/voices/samples", "public/cache/tts", "public/uploads/illustrations", "public/uploads/audio", "public/uploads/backgrounds", "public/uploads/voices"]:
    os.makedirs(os.path.join(target_dir, d), exist_ok=True)

# 6. Kiểm tra tùy chọn Google Drive
if CONNECT_GOOGLE_DRIVE:
    print("\n🔗 [GOOGLE DRIVE]: Đang kết nối Google Drive...")
    try:
        from google.colab import drive
        import shutil
        drive.mount("/content/drive")
        drive_folder = f"{repo_name} voice clone"
        drive_data_dir = f"/content/drive/MyDrive/{drive_folder}"
        drive_samples_dir = os.path.join(drive_data_dir, "samples")
        drive_channels_dir = os.path.join(drive_data_dir, "channels")
        os.makedirs(drive_samples_dir, exist_ok=True)
        os.makedirs(drive_channels_dir, exist_ok=True)
        local_voices_dir = os.path.join(target_dir, "data/voices")
        local_samples_dir = os.path.join(local_voices_dir, "samples")
        local_channels_dir = os.path.join(target_dir, "data/channels")
        os.environ["DRIVE_VOICES_DIR"] = drive_data_dir
        drive_meta = os.path.join(drive_data_dir, "cloned_voices.json")
        if os.path.exists(drive_meta):
            shutil.copy2(drive_meta, os.path.join(local_voices_dir, "cloned_voices.json"))
            count = 0
            for sf in os.listdir(drive_samples_dir):
                shutil.copy2(os.path.join(drive_samples_dir, sf), os.path.join(local_samples_dir, sf))
                count += 1
            print(f"   ✓ Đã tự động khôi phục {count} file giọng Clone từ Google Drive ({drive_folder})!")
        print(f"   ✓ Đã kết nối Drive: MyDrive/{drive_folder}/")
    except Exception as drive_err:
        print(f"   ⚠️ Chưa thể kết nối Google Drive: {drive_err}")
else:
    print("\n⚡ [CHẾ ĐỘ NHANH]: Không kết nối Google Drive (tiết kiệm thời gian, không cần cấp quyền).")
    print("👉 Khi nào cần Clone giọng & Lưu vào Drive, hãy bấm nút Play ▶️ tại BƯỚC 1.1 bên dưới bất kỳ lúc nào!")

print(f"\n🚀 Sẵn sàng! Thư mục dự án: {target_dir}")


In [ ]:
# @title 🎙️ BƯỚC 1.1 (Tùy chọn): NÚT KẾT NỐI GOOGLE DRIVE ĐỂ LƯU & DÙNG GIỌNG CLONE
# @markdown 💡 **LƯU Ý**: Nếu bạn chỉ dùng các giọng đọc có sẵn (**Microsoft Edge-TTS** hoặc **VieNeu AI mặc định**), **KHÔNG CẦN CHẠY CELL NÀY** để tránh mất thời gian cấp quyền Google Drive!
# @markdown 
# @markdown Bấm nút Play ▶️ của cell này khi bạn muốn:
# @markdown - Cấp quyền Google Drive để lưu giữ vĩnh viễn các file mẫu giọng clone (tự động tạo thư mục riêng: `<Tên dự án> voice clone`).
# @markdown - Khôi phục lại các giọng clone đã lưu từ các phiên làm việc trước vào Video Studio.
# @markdown 
# @markdown Chọn thao tác muốn thực hiện:
ACTION = "Kết nối Drive & Tải Giọng Clone vào Studio" # @param ["Kết nối Drive & Tải Giọng Clone vào Studio", "Sao lưu toàn bộ Giọng Clone hiện tại lên Google Drive", "Kiểm tra danh sách Giọng Clone trên Google Drive"]

import os
import shutil
import subprocess
import urllib.request
import json

PROJECT_DIR = os.environ.get("PROJECT_DIR", "/content/Video-quiz-new")
if not os.path.exists(PROJECT_DIR):
    for item in os.listdir("/content"):
        if os.path.isdir(f"/content/{item}") and os.path.exists(f"/content/{item}/package.json"):
            PROJECT_DIR = f"/content/{item}"
            break

repo_name = os.environ.get("PROJECT_NAME") or os.path.basename(PROJECT_DIR)
drive_folder_name = f"{repo_name} voice clone"
drive_voices_dir = f"/content/drive/MyDrive/{drive_folder_name}"
drive_samples_dir = os.path.join(drive_voices_dir, "samples")
drive_channels_dir = os.path.join(drive_voices_dir, "channels")

local_voices_dir = os.path.join(PROJECT_DIR, "data/voices")
local_samples_dir = os.path.join(local_voices_dir, "samples")
local_channels_dir = os.path.join(PROJECT_DIR, "data/channels")

os.makedirs(local_samples_dir, exist_ok=True)
os.makedirs(local_channels_dir, exist_ok=True)

print(f"🎙️ [VOICE CLONE MANAGER]: Đang thực hiện '{ACTION}'...")

# 1. Kết nối Google Drive
try:
    from google.colab import drive
    if not os.path.exists("/content/drive/MyDrive"):
        print("🔗 Đang mở hộp thoại cấp quyền kết nối Google Drive (nhấp 'Connect to Google Drive')...")
        drive.mount("/content/drive")
        print("✅ Đã kết nối Google Drive thành công!")
    else:
        print("✅ Google Drive đã được gắn kết sẵn sàng.")
except Exception as e:
    print(f"❌ Không thể kết nối Google Drive: {e}")
    print("👉 Vui lòng nhấp đồng ý cấp quyền khi Colab hiển thị thông báo.")
    raise

os.makedirs(drive_samples_dir, exist_ok=True)
os.makedirs(drive_channels_dir, exist_ok=True)
os.environ["DRIVE_VOICES_DIR"] = drive_voices_dir
os.environ["PROJECT_NAME"] = repo_name

# 2. Xử lý theo từng Action
if ACTION == "Kết nối Drive & Tải Giọng Clone vào Studio":
    count = 0
    drive_meta = os.path.join(drive_voices_dir, "cloned_voices.json")
    if os.path.exists(drive_meta):
        shutil.copy2(drive_meta, os.path.join(local_voices_dir, "cloned_voices.json"))
        for sf in os.listdir(drive_samples_dir):
            src_f = os.path.join(drive_samples_dir, sf)
            if os.path.isfile(src_f):
                shutil.copy2(src_f, os.path.join(local_samples_dir, sf))
                count += 1
        print(f"✅ Đã nạp thành công {count} file âm thanh giọng clone từ Drive vào Studio!")
    else:
        print(f"ℹ️ Thư mục Drive '{drive_folder_name}' sẵn sàng (chưa có giọng clone nào được lưu từ trước).")

    # Đồng bộ cấu hình kênh nếu có
    chan_count = 0
    if os.path.exists(drive_channels_dir):
        for cdir in os.listdir(drive_channels_dir):
            csrc = os.path.join(drive_channels_dir, cdir)
            if os.path.isdir(csrc):
                cdst = os.path.join(local_channels_dir, cdir)
                os.makedirs(cdst, exist_ok=True)
                for cf in os.listdir(csrc):
                    shutil.copy2(os.path.join(csrc, cf), os.path.join(cdst, cf))
                chan_count += 1
    if chan_count > 0:
        print(f"✅ Đã đồng bộ {chan_count} kênh cấu hình từ Google Drive!")

    # Nếu backend server đang chạy (port 5410), ping endpoint reload trực tiếp
    try:
        req = urllib.request.Request("http://127.0.0.1:5410/api/voices/drive/import", data=b"", method="POST")
        with urllib.request.urlopen(req, timeout=2) as resp:
            print("⚡ Đã cập nhật danh sách giọng clone trực tiếp vào Web Studio (không cần F5)!")
    except Exception:
        pass

    print(f"\n🎉 HOÀN TẤT KẾT NỐI GOOGLE DRIVE!")
    print(f"📁 Thư mục lưu trữ chuyên biệt: Google Drive > MyDrive > {drive_folder_name}")
    print(f"👉 Mọi giọng clone bạn tạo trong Studio từ giờ sẽ được bảo toàn vĩnh viễn trên Drive!")

elif ACTION == "Sao lưu toàn bộ Giọng Clone hiện tại lên Google Drive":
    local_meta = os.path.join(local_voices_dir, "cloned_voices.json")
    count = 0
    if os.path.exists(local_meta):
        shutil.copy2(local_meta, os.path.join(drive_voices_dir, "cloned_voices.json"))
        for sf in os.listdir(local_samples_dir):
            src_f = os.path.join(local_samples_dir, sf)
            if os.path.isfile(src_f):
                shutil.copy2(src_f, os.path.join(drive_samples_dir, sf))
                count += 1
        print(f"✅ Đã sao lưu thành công {count} file giọng clone lên Google Drive ({drive_folder_name})!")
    else:
        print("ℹ️ Hiện tại chưa có giọng clone nào trong Studio để sao lưu.")

elif ACTION == "Kiểm tra danh sách Giọng Clone trên Google Drive":
    drive_meta = os.path.join(drive_voices_dir, "cloned_voices.json")
    if os.path.exists(drive_meta):
        with open(drive_meta, "r", encoding="utf-8") as f:
            voices_list = json.load(f)
        print(f"📋 Danh sách {len(voices_list)} giọng clone đang lưu trên Google Drive ({drive_folder_name}):")
        for v in voices_list:
            print(f"   • {v.get('name', 'Không tên')} [{v.get('gender', 'N/A')}] - ID: {v.get('id', 'N/A')}")
    else:
        print(f"ℹ️ Thư mục Google Drive '{drive_folder_name}' hiện chưa có giọng clone nào.")


In [ ]:
# @title ⚙️ BƯỚC 2: Cài đặt Môi trường & Build ứng dụng (Rút gọn hiển thị)
import os
import subprocess
import re
import shutil

PROJECT_DIR = os.environ.get("PROJECT_DIR", "/content/Video-quiz-new")
if not os.path.exists(PROJECT_DIR):
    for item in os.listdir("/content"):
        if os.path.isdir(f"/content/{item}") and os.path.exists(f"/content/{item}/package.json"):
            PROJECT_DIR = f"/content/{item}"
            break
os.environ["PROJECT_DIR"] = PROJECT_DIR

print("⚙️ Đang tiến hành cài đặt môi trường Video Studio (Rút gọn log)...\n")

# 1. Node.js 20 & FFmpeg (GPU Driver)
need_node = True
try:
    node_res = subprocess.run("node -v", shell=True, capture_output=True, text=True, timeout=5)
    if node_res.returncode == 0:
        m = re.search(r'v?(\d+)', node_res.stdout.strip())
        if m and int(m.group(1)) >= 18:
            need_node = False
except Exception:
    pass

if need_node:
    subprocess.run("curl -fsSL --connect-timeout 10 --max-time 30 https://deb.nodesource.com/setup_20.x | bash - > /dev/null 2>&1", shell=True)
    subprocess.run("apt-get install -y -qq -o DPkg::Lock::Timeout=60 nodejs > /dev/null 2>&1", shell=True)

subprocess.run("apt-get update -qq > /dev/null 2>&1 && apt-get install -y -qq -o DPkg::Lock::Timeout=60 ffmpeg libegl1-mesa libgl1-mesa-glx libgl1-mesa-dri python-is-python3 > /dev/null 2>&1", shell=True)
node_ver = subprocess.run("node -v", shell=True, capture_output=True, text=True).stdout.strip()
print(f"✅ [1/5] Node.js ({node_ver}) & FFmpeg (GPU Driver): Đã cài đặt xong!")

# 2. Font chữ tiếng Việt
subprocess.run("""
apt-get update -qq > /dev/null 2>&1 && apt-get install -y -qq -o DPkg::Lock::Timeout=60 fonts-noto fonts-liberation fonts-dejavu-core fonts-roboto fontconfig > /dev/null 2>&1
mkdir -p /usr/share/fonts/truetype/vietnamese
wget -q -nc -O /usr/share/fonts/truetype/vietnamese/Roboto-Regular.ttf "https://fonts.gstatic.com/s/roboto/v32/KFOlCnqEu92Fr1MmEU9fBBc4.ttf"
wget -q -nc -O /usr/share/fonts/truetype/vietnamese/Roboto-Bold.ttf "https://fonts.gstatic.com/s/roboto/v32/KFOlCnqEu92Fr1MmWUlfBBc4.ttf"
wget -q -nc -O /usr/share/fonts/truetype/vietnamese/Nunito-Regular.ttf "https://fonts.gstatic.com/s/nunito/v26/XRXV3I6Li01BKofINeaB.ttf"
wget -q -nc -O /usr/share/fonts/truetype/vietnamese/Nunito-Bold.ttf "https://fonts.gstatic.com/s/nunito/v26/XRXW3I6Li01BKofA-sukbvhV.ttf"
wget -q -nc -O /usr/share/fonts/truetype/vietnamese/Montserrat-Regular.ttf "https://fonts.gstatic.com/s/montserrat/v31/JTUHjIg1_i6t8kCHKm4532VJOt5-QNFgpCtr6Ew-.ttf"
wget -q -nc -O /usr/share/fonts/truetype/vietnamese/Montserrat-Bold.ttf "https://fonts.gstatic.com/s/montserrat/v31/JTUHjIg1_i6t8kCHKm4532VJOt5-QNFgpCuM70w-.ttf"
wget -q -nc -O /usr/share/fonts/truetype/vietnamese/Lora-Regular.ttf "https://fonts.gstatic.com/s/lora/v37/0QI6MX1D_JOuGQbT0gvTJPa787weuyJG.ttf"
wget -q -nc -O /usr/share/fonts/truetype/vietnamese/Lora-Bold.ttf "https://fonts.gstatic.com/s/lora/v37/0QI6MX1D_JOuGQbT0gvTJPa787z5vCJG.ttf"
wget -q -nc -O /usr/share/fonts/truetype/vietnamese/BeVietnamPro-Regular.ttf "https://fonts.gstatic.com/s/bevietnampro/v11/QFdG35STV5CdGTuE6vtGWC3Q.ttf"
wget -q -nc -O /usr/share/fonts/truetype/vietnamese/BeVietnamPro-Bold.ttf "https://fonts.gstatic.com/s/bevietnampro/v11/QFdG35STV5CdGTuE6vtGWC2Z.ttf"
fc-cache -fv > /dev/null 2>&1
""", shell=True)
print("✅ [2/5] Font chữ tiếng Việt (Montserrat, Roboto, Nunito, Lora...): Đã cài đặt xong!")

# 3. VieNeu AI v3 & Microsoft Edge-TTS
subprocess.run("pip install -q vieneu soundfile librosa edge-tts > /dev/null 2>&1", shell=True)
subprocess.run("which edge-tts && ln -sf $(which edge-tts) /usr/local/bin/edge-tts 2>/dev/null || true", shell=True, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
subprocess.run("which edge-tts && ln -sf $(which edge-tts) /usr/bin/edge-tts 2>/dev/null || true", shell=True, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
try:
    from vieneu import Vieneu
    _test = Vieneu()
except Exception:
    pass
print("✅ [3/5] Giọng đọc VieNeu AI v3 & Microsoft Edge-TTS: Đã cài đặt xong!")

# 4. Cloudflared Tunnel
subprocess.run("wget -q -nc https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb > /dev/null 2>&1", shell=True)
subprocess.run("dpkg -i cloudflared-linux-amd64.deb > /dev/null 2>&1", shell=True)
print("✅ [4/5] Đường truyền Cloudflared Tunnel (HTTPS miễn phí): Đã cài đặt xong!")

# 5. npm dependencies & Build Studio (ẩn log rác)
subprocess.run(f"cd '{PROJECT_DIR}' && npm install --prefer-offline --no-audit --loglevel=error > /dev/null 2>&1", shell=True)
subprocess.run(f"cd '{PROJECT_DIR}' && npm run build > /dev/null 2>&1", shell=True)
subprocess.run(f"chmod +x '{PROJECT_DIR}'/node_modules/@remotion/compositor-linux-x64-gnu/ffmpeg 2>/dev/null || true", shell=True)
print("✅ [5/5] Cài đặt thư viện & Biên dịch Studio (Vite + TypeScript): Đã cài đặt xong!")

# Kiểm tra GPU & phần cứng NVENC
if shutil.which("nvidia-smi"):
    try:
        smi_out = subprocess.check_output(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"]).decode().strip()
        print(f"\n🔥 [GPU ĐÃ KÍCH HOẠT]: {smi_out} (Hỗ trợ render NVENC 25 - 35+ FPS)")
    except Exception:
        pass

print("\n🎉 CÀI ĐẶT MÔI TRƯỜNG THÀNH CÔNG! Hãy bấm nút Chạy ▶️ ở BƯỚC 3 để mở Web Studio.")


In [ ]:
# @title 🚀 BƯỚC 3: Khởi động Web App & Mở Link HTTPS Cloudflare Tunnel
import os
import subprocess
import time
import re
import socket
import urllib.request
import urllib.error
from IPython.display import display, HTML

# 1. Xác định thư mục dự án
PROJECT_DIR = os.environ.get("PROJECT_DIR", "/content/Video-quiz-new")
if not os.path.exists(PROJECT_DIR):
    for item in os.listdir("/content"):
        if os.path.isdir(f"/content/{item}") and os.path.exists(f"/content/{item}/package.json"):
            PROJECT_DIR = f"/content/{item}"
            break

os.environ["PROJECT_DIR"] = PROJECT_DIR
SERVER_PORT = 5410

# 2. Dọn dẹp sạch sẽ các process cũ để giải phóng triệt để port 5410, 4500, 3001
print(f"🧹 Đang dọn dẹp các tiến trình cũ và giải phóng port {SERVER_PORT}...")
subprocess.run(f"fuser -k {SERVER_PORT}/tcp 4500/tcp 3001/tcp > /dev/null 2>&1 || true", shell=True)
subprocess.run("pkill -9 -f 'node.*backend' || true", shell=True)
subprocess.run("pkill -9 -f 'tsx.*backend' || true", shell=True)
subprocess.run("pkill -9 -f 'vite' || true", shell=True)
subprocess.run("pkill -9 -f 'cloudflared' || true", shell=True)
time.sleep(2)

# Helper kiểm tra đa luồng (HTTP + Socket fallback trên cả 127.0.0.1 & localhost & 0.0.0.0)
def check_service(port, path="", timeout=1.5):
    for host in ["127.0.0.1", "localhost", "0.0.0.0"]:
        url = f"http://{host}:{port}{path}"
        try:
            req = urllib.request.Request(
                url,
                headers={'User-Agent': 'ColabHealthCheck', 'Host': f'localhost:{port}'}
            )
            with urllib.request.urlopen(req, timeout=timeout) as resp:
                if resp.status in (200, 304):
                    return True
        except urllib.error.HTTPError as he:
            if he.code in (200, 304):
                return True
        except Exception:
            pass
        try:
            with socket.create_connection((host, port), timeout=timeout):
                return True
        except Exception:
            pass
    return False

# Đảm bảo Edge-TTS & Cloudflared đã cài đặt sẵn sàng
if subprocess.run("which edge-tts > /dev/null 2>&1", shell=True).returncode != 0:
    print("⏳ Đang cài đặt nhanh Microsoft Edge-TTS...")
    subprocess.run("pip install -q edge-tts && ln -sf $(which edge-tts) /usr/local/bin/edge-tts 2>/dev/null || true", shell=True)

# Đảm bảo Cloudflared đã cài đặt
if subprocess.run("which cloudflared > /dev/null 2>&1", shell=True).returncode != 0:
    print("⏳ Đang cài đặt nhanh Cloudflared Tunnel...")
    subprocess.run("wget -q -nc https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb && dpkg -i cloudflared-linux-amd64.deb > /dev/null 2>&1", shell=True)

# Đảm bảo frontend đã có bản build production bundle
dist_index = os.path.join(PROJECT_DIR, "frontend/dist/index.html")
root_dist_index = os.path.join(PROJECT_DIR, "dist/index.html")
if not os.path.exists(dist_index) and not os.path.exists(root_dist_index):
    print("⚙️ Đang biên dịch Production Bundle cho Frontend & Backend...")
    subprocess.run(f"cd '{PROJECT_DIR}' && npm run build", shell=True)

# 3. Khởi động Production All-in-one Server (Port 5410)
# Express phục vụ trực tiếp Web App + Toàn bộ API + Native Skia Render + TTS + Xuất video MP4
# Không qua proxy trung gian -> Triệt tiêu hoàn toàn lỗi tự reload trang, tối ưu RAM & CPU Colab!
print(f"\n⚙️ [1/2] Đang khởi động Production Server (Port {SERVER_PORT} - Web App & Studio API)...\n")
backend_log = "/content/backend.log"
with open(backend_log, "w", encoding="utf-8") as f:
    f.write("=== VIDEO QUIZ STUDIO BACKEND LOG ===\n")

backend_proc = subprocess.Popen(
    f"cd '{PROJECT_DIR}' && PORT={SERVER_PORT} DRIVE_VOICES_DIR='{os.environ.get("DRIVE_VOICES_DIR", "")}' PROJECT_NAME='{os.environ.get("PROJECT_NAME", "")}' npx tsx --tsconfig backend/tsconfig.json backend/src/index.ts",
    shell=True,
    stdout=open(backend_log, "a"),
    stderr=subprocess.STDOUT
)

# Health Check Backend (Port 5410)
backend_ready = False
for i in range(30):
    time.sleep(1)
    if backend_proc.poll() is not None:
        print("\n❌ LỖI: Backend process đã dừng đột ngột!")
        break
    if check_service(SERVER_PORT, "/api/system/health") or check_service(SERVER_PORT, "/api/health"):
        backend_ready = True
        break

if not backend_ready:
    print("\n" + "!"*70)
    print(f"❌ LỖI: Backend không thể khởi động trên port {SERVER_PORT}!")
    print("📋 CHI TIẾT LOG BACKEND (/content/backend.log):")
    print("-" * 70)
    if os.path.exists(backend_log):
        with open(backend_log, "r", encoding="utf-8", errors="ignore") as f:
            print(f.read())
    print("-" * 70)
    raise RuntimeError("Backend startup failed. Dừng quy trình.")
else:
    print(f"✅ Server đã sẵn sàng và phản hồi tốt tại http://127.0.0.1:{SERVER_PORT} (Health: OK)!")

# 4. Khởi động Cloudflare Tunnel trỏ trực tiếp vào Server (Port 5410)
print(f"\n🌐 [2/2] Đang khởi tạo Cloudflare Tunnel trỏ trực tiếp vào Server (http://127.0.0.1:{SERVER_PORT})...\n")
tunnel_log = "/content/tunnel.log"
if os.path.exists(tunnel_log):
    os.remove(tunnel_log)

tunnel_proc = subprocess.Popen(
    f"cloudflared tunnel --url http://127.0.0.1:{SERVER_PORT}",
    shell=True,
    stdout=open(tunnel_log, "w"),
    stderr=subprocess.STDOUT
)

# 5. Đọc URL HTTPS công khai từ Cloudflared
public_url = None
for _ in range(40):
    time.sleep(1)
    if os.path.exists(tunnel_log):
        with open(tunnel_log, "r", encoding="utf-8", errors="ignore") as f:
            content = f.read()
            matches = re.findall(r"https://[a-zA-Z0-9-]+\.trycloudflare\.com", content)
            if matches:
                public_url = matches[0]
                break

if not public_url:
    print("\n⚠️ Không tìm thấy URL Cloudflare Tunnel. Log tunnel:")
    if os.path.exists(tunnel_log):
        with open(tunnel_log, "r", encoding="utf-8", errors="ignore") as f:
            print(f.read()[-500:])
    backend_proc.terminate()
    raise RuntimeError("Không lấy được Cloudflare Tunnel URL.")

print("\n" + "="*70)
print("🎉 VIDEO QUIZ STUDIO ĐÃ SẴN SÀNG HOẠT ĐỘNG (100% TRUY CẬP ĐƯỢC)!")
print(f"👉 Bấm vào liên kết này để mở Web App: {public_url}")
print("="*70 + "\n")

display(HTML(f"""
<div style="background: linear-gradient(135deg, #0f172a, #1e293b); border: 2px solid #3b82f6; border-radius: 16px; padding: 24px; color: white; font-family: -apple-system, BlinkMacSystemFont, 'Segoe UI', Roboto, sans-serif; text-align: center; max-width: 620px; margin: 16px auto; box-shadow: 0 12px 30px -5px rgba(59, 130, 246, 0.35);">
    <div style="font-size: 40px; margin-bottom: 8px;">🎬 🎯 ✨</div>
    <h2 style="color: #60a5fa; margin: 0 0 8px 0; font-size: 22px; font-weight: 800; letter-spacing: 0.5px;">VIDEO QUIZ STUDIO</h2>
    <p style="color: #94a3b8; font-size: 13px; margin-bottom: 22px; line-height: 1.5;">Ứng dụng chạy trực tiếp trên Server Production (Port {SERVER_PORT}) qua Cloudflare Tunnel tốc độ cao, không gián đoạn kết nối. Bấm nút bên dưới để sử dụng:</p>
    <a href="{public_url}" target="_blank" style="display: inline-block; background: linear-gradient(90deg, #2563eb, #4f46e5); color: white; padding: 14px 32px; border-radius: 12px; font-weight: 700; font-size: 16px; text-decoration: none; box-shadow: 0 4px 16px rgba(37, 99, 235, 0.45); transition: all 0.2s;">
        👉 BẤM VÀO ĐÂY ĐỂ MỞ ỨNG DỤNG 👈
    </a>
    <div style="margin-top: 18px; font-size: 12px; color: #64748b;">
        ⚡ Đường truyền mã hóa Cloudflare HTTPS • Không tự reload • Giữ tab Colab này để duy trì kết nối
    </div>
</div>
"""))

# 6. Giữ tiến trình chạy ổn định & giám sát liên tục (Keep-alive monitor)
print("📡 Đang duy trì kết nối Web App... (Nhấn nút Dừng ■ ở cạnh cell khi muốn dừng tool)\n")
try:
    heartbeat = 0
    while True:
        time.sleep(10)
        heartbeat += 1
        if backend_proc.poll() is not None:
            print("⚠️ Cảnh báo: Backend server đã dừng!")
            break
        if tunnel_proc.poll() is not None:
            print("⚠️ Cảnh báo: Cloudflare Tunnel đã dừng!")
            break
        if heartbeat % 30 == 0:  # mỗi 5 phút
            print(f"⏳ Phiên hoạt động đang chạy tốt... (URL: {public_url})")
except KeyboardInterrupt:
    print("\n🛑 Đã nhận lệnh dừng từ người dùng. Đang tắt các tiến trình...")
    backend_proc.terminate()
    tunnel_proc.terminate()
    print("✅ Đã dọn dẹp sạch sẽ.")


---  
## 🛠️ CÁC TIỆN ÍCH QUẢN LÝ VIDEO, XEM LOG & CẬP NHẬT CODE

In [ ]:
# @title 📦 1. Tải toàn bộ video đã Render về máy (File ZIP hoặc Lưu Google Drive)
import os
import zipfile
from google.colab import files

PROJECT_DIR = os.environ.get("PROJECT_DIR", "/content/Video-quiz-new")
if not os.path.exists(PROJECT_DIR):
    for item in os.listdir("/content"):
        if os.path.isdir(f"/content/{item}") and os.path.exists(f"/content/{item}/package.json"):
            PROJECT_DIR = f"/content/{item}"
            break

# Kiểm tra cả hai thư mục render khả thi (data/renders và output)
check_dirs = [
    os.path.join(PROJECT_DIR, "data/renders"),
    os.path.join(PROJECT_DIR, "output")
]

# Tùy chọn lưu vào Google Drive
SAVE_TO_GOOGLE_DRIVE = False # @param {type:"boolean"}
DRIVE_FOLDER_NAME = "Quiz_Videos_Exported" # @param {type:"string"}

mp4_files_with_path = []
for cdir in check_dirs:
    if os.path.exists(cdir):
        for f in os.listdir(cdir):
            if f.endswith(".mp4"):
                mp4_files_with_path.append(os.path.join(cdir, f))

if not mp4_files_with_path:
    print(f"⚠️ Chưa có video MP4 nào trong thư mục render ({check_dirs[0]})")
    print("👉 Hãy tạo và xuất video trên giao diện web trước rồi quay lại chạy ô này nhé!")
else:
    print(f"🎬 Đã tìm thấy {len(mp4_files_with_path)} video MP4:")
    total_size = 0
    for fpath in sorted(mp4_files_with_path):
        fname = os.path.basename(fpath)
        size_mb = os.path.getsize(fpath) / (1024 * 1024)
        total_size += size_mb
        print(f"  ✅ {fname} ({size_mb:.2f} MB)")
    print(f"📊 Tổng dung lượng: {total_size:.2f} MB\n")

    if SAVE_TO_GOOGLE_DRIVE:
        from google.colab import drive
        import shutil
        print("🔗 Đang kết nối Google Drive...")
        drive.mount("/content/drive")
        dest_drive = f"/content/drive/MyDrive/{DRIVE_FOLDER_NAME}"
        os.makedirs(dest_drive, exist_ok=True)
        for fpath in mp4_files_with_path:
            fname = os.path.basename(fpath)
            shutil.copy2(fpath, os.path.join(dest_drive, fname))
        print(f"\n🎉 ĐÃ LƯU XONG {len(mp4_files_with_path)} VIDEO VÀO GOOGLE DRIVE!")
        print(f"📁 Đường dẫn trên Drive của bạn: MyDrive/{DRIVE_FOLDER_NAME}/")
    else:
        zip_path = "/content/Quiz_Videos_Exported.zip"
        with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as zipf:
            for fpath in mp4_files_with_path:
                fname = os.path.basename(fpath)
                zipf.write(fpath, arcname=fname)
        zip_mb = os.path.getsize(zip_path) / (1024 * 1024)
        print(f"🚀 Đang bắt đầu tải file ZIP về máy ({zip_mb:.2f} MB)...\n")
        files.download(zip_path)


In [ ]:
# @title 📋 2. Xem Log Server (Kiểm tra tiến trình Backend / Tunnel)
import os

print("="*60)
print("📂 1. LOG BACKEND SERVER (/content/backend.log):")
print("="*60)
if os.path.exists("/content/backend.log"):
    with open("/content/backend.log", "r") as f:
        lines = f.readlines()
        print("".join(lines[-40:]))
else:
    print("⚠️ Chưa có file log backend.")

print("\n" + "="*60)
print("📂 2. LOG CLOUDFLARE TUNNEL (/content/tunnel.log):")
print("="*60)
if os.path.exists("/content/tunnel.log"):
    with open("/content/tunnel.log", "r") as f:
        lines = f.readlines()
        print("".join(lines[-20:]))
else:
    print("⚠️ Chưa có file log tunnel.")


In [ ]:
# @title 🔄 3. Cập nhật mã nguồn mới nhất (Git Pull) & Khởi động lại
import os
import subprocess
import time

PROJECT_DIR = os.environ.get("PROJECT_DIR", "/content/Video-quiz-new")
if not os.path.exists(PROJECT_DIR):
    for item in os.listdir("/content"):
        if os.path.isdir(f"/content/{item}") and os.path.exists(f"/content/{item}/package.json"):
            PROJECT_DIR = f"/content/{item}"
            break

print("📥 Đang kéo commit mới nhất từ GitHub...")
subprocess.run(f"cd '{PROJECT_DIR}' && git pull origin main || git pull origin master", shell=True)

print("⚙️ Build lại Frontend & Backend...")
subprocess.run(f"cd '{PROJECT_DIR}' && npm run build", shell=True)

print("🧹 Dọn dẹp tiến trình cũ...")
subprocess.run("fuser -k 5410/tcp 4500/tcp 3001/tcp > /dev/null 2>&1 || true", shell=True)
subprocess.run("pkill -9 -f 'node.*backend' || true", shell=True)
subprocess.run("pkill -9 -f 'tsx.*backend' || true", shell=True)
subprocess.run("pkill -9 -f 'vite' || true", shell=True)
subprocess.run("pkill -9 -f 'cloudflared' || true", shell=True)
time.sleep(2)

print("\n✅ Đã cập nhật xong! Hãy bấm nút ▶️ ở BƯỚC 3 để khởi động lại Web App và Tunnel mới.")


In [ ]:
# @title 💾 4. Sao lưu & Khôi phục Giọng Clone (Google Drive hoặc Tải file ZIP)
import os
import shutil
import zipfile
from google.colab import files

PROJECT_DIR = os.environ.get("PROJECT_DIR", "/content/Video-quiz-new")
if not os.path.exists(PROJECT_DIR):
    for item in os.listdir("/content"):
        if os.path.isdir(f"/content/{item}") and os.path.exists(f"/content/{item}/package.json"):
            PROJECT_DIR = f"/content/{item}"
            break

repo_name = os.environ.get("PROJECT_NAME") or os.path.basename(PROJECT_DIR)
drive_folder_name = f"{repo_name} voice clone"
drive_voices = f"/content/drive/MyDrive/{drive_folder_name}"

voices_dir = os.path.join(PROJECT_DIR, "data/voices")
samples_dir = os.path.join(voices_dir, "samples")
cloned_json = os.path.join(voices_dir, "cloned_voices.json")

ACTION = "Lưu vào Google Drive" # @param ["Lưu vào Google Drive", "Tải file ZIP về máy tính", "Khôi phục từ Google Drive"]

if ACTION == "Lưu vào Google Drive":
    from google.colab import drive
    if not os.path.exists("/content/drive/MyDrive"):
        drive.mount("/content/drive")
    os.makedirs(os.path.join(drive_voices, "samples"), exist_ok=True)
    if os.path.exists(cloned_json):
        shutil.copy2(cloned_json, drive_voices)
    count = 0
    if os.path.exists(samples_dir):
        for f in os.listdir(samples_dir):
            shutil.copy2(os.path.join(samples_dir, f), os.path.join(drive_voices, "samples", f))
            count += 1
    print(f"🎉 Đã lưu toàn bộ {count} giọng Clone vào Google Drive: MyDrive/{drive_folder_name}/ thành công!")

elif ACTION == "Tải file ZIP về máy tính":
    zip_path = f"/content/{repo_name}_Cloned_Voices.zip"
    with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as zipf:
        if os.path.exists(cloned_json):
            zipf.write(cloned_json, arcname="cloned_voices.json")
        if os.path.exists(samples_dir):
            for f in os.listdir(samples_dir):
                zipf.write(os.path.join(samples_dir, f), arcname=f"samples/{f}")
    print("🚀 Đang tải file ZIP về máy...")
    files.download(zip_path)

elif ACTION == "Khôi phục từ Google Drive":
    from google.colab import drive
    if not os.path.exists("/content/drive/MyDrive"):
        drive.mount("/content/drive")
    count = 0
    if os.path.exists(os.path.join(drive_voices, "cloned_voices.json")):
        shutil.copy2(os.path.join(drive_voices, "cloned_voices.json"), voices_dir)
        for f in os.listdir(os.path.join(drive_voices, "samples")):
            shutil.copy2(os.path.join(drive_voices, "samples", f), os.path.join(samples_dir, f))
            count += 1
        print(f"🎉 Đã khôi phục thành công {count} file giọng Clone từ Google Drive ({drive_folder_name})!")
    else:
        print(f"⚠️ Chưa tìm thấy dữ liệu giọng clone trong MyDrive/{drive_folder_name}.")
